In [ ]:
# SETUP
# ==========================================

from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Install Dependencies

print("Installing OpenSlide and dependencies...")
!apt-get install -y openslide-tools > /dev/null
!pip install openslide-python torch torchvision h5py opencv-python-headless > /dev/null

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import openslide
import numpy as np
import cv2 # Required for your tissue detection logic
import h5py
from PIL import Image

# 3. Define Paths
#------------------------------------------
PROJECT_DIR = '/content/drive/MyDrive'
WEIGHTS_PATH = os.path.join(PROJECT_DIR, 'KimiaNetPyTorchWeights.pth')
#------------------------------------------

# Check if GPU is on (Required for KimiaNet)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Setup Complete. Using device: {device}")

Mounted at /content/drive
Installing OpenSlide and dependencies...
✅ Setup Complete. Using device: cpu


In [ ]:
# MODEL DEFINITION (From KimiaNet_PyTorch_Feature_Extraction.py)
# ========================================================================
import torch
import torch.nn as nn
import torchvision.models as models
import os

class fully_connected(nn.Module):
    """Custom layer to extract the 1024 feature vector"""
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        # We only return the flattened features for AMIL
        return x

def get_kimianet_model(weights_path):
    print("Initializing KimiaNet (DenseNet-121)...")

    # 1. Load base DenseNet
    model = models.densenet121(pretrained=True)

    # 2. Freeze weights so we don't accidentally train it
    for param in model.parameters():
        param.requires_grad = False

    # 3. Add pooling layer to force output to 1x1
    model.features = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(output_size=(1, 1)))
    num_ftrs = model.classifier.in_features

    # 4. Wrap in custom class (30 was the original KimiaNet class count)
    model_final = fully_connected(model.features, num_ftrs, 30)
    model_final = model_final.to(device)

    # 5. Handle DataParallel (Required because your .pth was saved on multiple GPUs)
    model_final = nn.DataParallel(model_final)

    # 6. Load your custom weights
    if os.path.exists(weights_path):
        print(f"Loading weights from: {weights_path}")
        state_dict = torch.load(weights_path, map_location=device)
        model_final.load_state_dict(state_dict)
        print("✅ Weights loaded successfully!")
    else:
        print("⚠️ WARNING: Weights file not found!")

    return model_final

In [ ]:
# ======================================
# PREPROCESSING
# ======================================

import os
import glob
import h5py
import cv2
import numpy as np
import openslide
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# fall back (read aperio.AppMag)
def get_base_magnification(slide, slide_path=None, default_mag=20.0):
    """
    Safely get base magnification from common WSI metadata fields.
    """

    # 1. Standard OpenSlide key
    objective_power = slide.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER)
    if objective_power is not None:
        return float(objective_power), "openslide objective-power"

    # 2. Aperio SVS key
    aperio_mag = slide.properties.get("aperio.AppMag")
    if aperio_mag is not None:
        return float(aperio_mag), "aperio.AppMag"

    # 3. No metadata
    return None, "missing metadata"

class Unified_WSI_Dataset(Dataset):

    def __init__(self, slide_path, target_patch_size=512, resize_to_224=False):
        self.slide_path = slide_path
        self.slide = openslide.OpenSlide(slide_path)
        self.target_patch_size = target_patch_size
        self.resize_to_224 = resize_to_224
        self.to_tensor = transforms.ToTensor()


        # TARGET MAGNIFICATION
        self.target_mag = 5.0

        # Read slide metadata safely
        self.base_mag, self.mag_source = get_base_magnification(
          self.slide,
          slide_path=self.slide_path
        )

        if self.base_mag is None:
          raise ValueError(
            f"Missing reliable magnification metadata. Source: {self.mag_source}"
          )

        # How much downsampling is needed from level 0 to target magnification
        self.required_downsample = self.base_mag / self.target_mag


        level_downsamples = np.array(self.slide.level_downsamples, dtype=np.float32)

        # Prefer the highest-resolution level that does not exceed the required downsample

        tolerance = 0.05 # allow tiny rounding differences

        candidate_levels = np.where(level_downsamples <= self.required_downsample  + tolerance )[0]

        if len(candidate_levels) > 0:
            self.level = int(candidate_levels[-1])
        else:
            self.level =  int(np.argmin(np.abs(level_downsamples - self.required_downsample)))


        self.level_downsample = float(self.slide.level_downsamples[self.level])

        # True level-0 field of view needed for a 512x512 patch at target magnification
        self.step_at_level_0 = int(round(self.target_patch_size * self.required_downsample))

        # How large the patch must be read at the chosen level before resizing to 512
        self.read_size_at_level = int(round(self.step_at_level_0 / self.level_downsample))

        effective_mag_if_read_direct = self.base_mag / self.level_downsample

        print(f"Slide: {os.path.basename(slide_path)}")
        print()
        print(f"  Base magnification: {self.base_mag}x ({self.mag_source})")
        print(f"  Target magnification: {self.target_mag}x")
        print(f"  Required downsample: {self.required_downsample}")
        print(f"  Chosen level: {self.level}")
        print(f"  Level downsample: {self.level_downsample}")
        print(f"  Direct magnification at chosen level: {effective_mag_if_read_direct:.4f}x")
        print(f"  Target field-of-view at level 0: {self.step_at_level_0} x {self.step_at_level_0}")
        print(f"  Read size at chosen level before resize: {self.read_size_at_level} x {self.read_size_at_level}")
        print(f"  Final output patch size: {self.target_patch_size} x {self.target_patch_size}")
        print(f"  Level dimensions: {self.slide.level_dimensions}")
        print(f"  Level downsamples: {self.slide.level_downsamples}")
        print()

        # Standard normalization
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.coords = self._scan_and_filter()

    def _scan_and_filter(self):
        w, h = self.slide.dimensions
        downsample_thumb = 32
        thumb = self.slide.get_thumbnail((w//downsample_thumb, h//downsample_thumb)).convert("RGB")
        thumb_np = np.array(thumb)

        gray_img = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2GRAY)
        _, im_bw = cv2.threshold(gray_img, 200, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((5, 5), np.uint8)
        im_bw = cv2.morphologyEx(im_bw, cv2.MORPH_CLOSE, kernel)
        tissue_mask = (im_bw > 0).astype(np.uint8)

        valid_patches = []
        thumb_step = max(1, self.step_at_level_0 // downsample_thumb)

        for y in range(0, tissue_mask.shape[0], thumb_step):
            for x in range(0, tissue_mask.shape[1], thumb_step):
                patch_mask = tissue_mask[y:y+thumb_step, x:x+thumb_step]

                # Keep if patch is at least 20% tissue
                if np.mean(patch_mask) >= 0.20:
                    real_x, real_y = x * downsample_thumb, y * downsample_thumb
                    if real_x + self.step_at_level_0 <= w and real_y + self.step_at_level_0 <= h:
                        valid_patches.append((real_x, real_y))

        print(f"  Slide dimensions (level 0): {w} x {h}")
        print(f"  Thumbnail downsample used for tissue mask: {downsample_thumb}")
        print(f"  Thumbnail step: {thumb_step}")

        approx_grid_x = w // self.step_at_level_0
        approx_grid_y = h // self.step_at_level_0
        print(f"  Approx max non-overlapping grid at target FOV: {approx_grid_x} x {approx_grid_y}")
        print()
        print(f"    🔍 Found {len(valid_patches)} valid tissue patches at 5x zoom.")
        return valid_patches

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
            x, y = self.coords[idx]

            # Read directly from the chosen pyramid level at the target patch size
            img = self.slide.read_region(
                (x, y),
                self.level,
                (self.read_size_at_level, self.read_size_at_level)
            ).convert("RGB")

            if self.read_size_at_level != self.target_patch_size:
                img = img.resize((self.target_patch_size, self.target_patch_size))

            if self.resize_to_224:
                img = img.resize((224, 224))

            img_tensor = self.to_tensor(img)
            img_tensor = self.normalize(img_tensor)
            return img_tensor, np.array([x, y])

# ======================================
# PARALLEL EXTRACTION
# ======================================
import shutil
import os
import pandas as pd

def extract_features_parallel(wsi_folder, output_folder, model, start_idx=0, end_idx=None, batch_size=32, num_workers=2, resize_to_224=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    all_slides = sorted(glob.glob(os.path.join(wsi_folder, '*.svs')))
    if end_idx is None: end_idx = len(all_slides)
    chunk_slides = all_slides[start_idx:end_idx]

    print("="*50)
    print(f"🚀 PARALLEL EXTRACTION STARTED")
    print(f"Processing Chunk: Slide {start_idx} to {end_idx-1} ({len(chunk_slides)} total slides)")
    print("="*50)

    os.makedirs(output_folder, exist_ok=True)

    skipped_slides = []

    for slide_idx, slide_path in enumerate(chunk_slides):
        slide_name = os.path.basename(slide_path)
        print(f"\n[{slide_idx + 1}/{len(chunk_slides)}] Processing... {slide_name}")

        output_h5_path = os.path.join(output_folder, f"{slide_name}_features.h5")
        if os.path.exists(output_h5_path):
            print(f"⏩ Already processed. Skipping...")
            continue

        # ==========================================================
        #  Copy file to Colab's fast local SSD
        # ==========================================================
        local_svs_path = f"/content/{slide_name}"

        try:
          print("  📥 Copying slide to local SSD...")
          shutil.copy(slide_path, local_svs_path)

          # Use the LOCAL path for the dataset, not the Drive path
          dataset = Unified_WSI_Dataset(local_svs_path, resize_to_224=resize_to_224)

          if len(dataset) == 0:
              print("  ⚠️ No tissue found. Skipping...")
              os.remove(local_svs_path) # Clean up SSD
              continue

          # Keep workers at 2 for Colab CPU limits
          loader = DataLoader(
              dataset,
              batch_size=batch_size,
              shuffle=False,
              num_workers=num_workers,
              pin_memory=torch.cuda.is_available()
          )

          all_features, all_coords = [], []

          with torch.no_grad():
              for batch_idx, (images, coords) in enumerate(loader):
                  images = images.to(device)
                  features = model(images)
                  all_features.append(features.cpu().numpy())
                  all_coords.append(coords.numpy())
                  if batch_idx % 10 == 0:
                      print(f"    Processed batch {batch_idx}/{len(loader)}")

          final_features = np.concatenate(all_features, axis=0)
          final_coords = np.concatenate(all_coords, axis=0)

          with h5py.File(output_h5_path, 'w') as f:
              f.create_dataset('features', data=final_features)
              f.create_dataset('coords', data=final_coords)

          print(f"    ✅ Saved to {output_h5_path}")

        except ValueError as e:
            reason = str(e)
            print(f"  ⚠️ Skipping {slide_name}: {reason}")
            skipped_slides.append({
                "slide_name": slide_name,
                "slide_path": slide_path,
                "reason": reason
            })

        finally:
            if os.path.exists(local_svs_path):
                os.remove(local_svs_path)


    if skipped_slides:
            skipped_df = pd.DataFrame(skipped_slides)

            print(f"\n⚠️ Skipped {len(skipped_slides)} slides:")
            for item in skipped_slides:
                print(f"- {item['slide_name']} | {item['reason']}")
    else:
            print("\n🎉 No slides were skipped.")

In [ ]:
# =========================================
# EXECUTION CELL
# =========================================

# 1. Define slides path, and where to save the .h5 files

#------------------------------------------
WSI_FOLDER_PATH = '/content/drive/MyDrive/CPTAC-LUAD/TUMOR'
OUTPUT_FOLDER_PATH = '/content/drive/MyDrive/CPTAC-LUAD_kimianet_features/TUMOR'
#------------------------------------------


# 2. Load specific model (KimiaNet here)
kimianet_model = get_kimianet_model(WEIGHTS_PATH)

# 3. RUN THE PIPELINE!
extract_features_parallel(
    wsi_folder=WSI_FOLDER_PATH,
    output_folder=OUTPUT_FOLDER_PATH,
    model=kimianet_model,
    start_idx=400,
    end_idx=402,
    batch_size=32,      # 32 is safe for Colab T4 GPUs
    num_workers=2,      # suggested max number of worker in current system is 2 (excessive worker creation might get DataLoader running slow or even freeze)
    resize_to_224=False  # Keep False for KimiaNet
)

Initializing KimiaNet (DenseNet-121)...
Loading weights from: /content/drive/MyDrive/KimiaNetPyTorchWeights.pth
✅ Weights loaded successfully!
🚀 PARALLEL EXTRACTION STARTED
Processing Chunk: Slide 400 to 401 (2 total slides)

[1/2] Processing... C3N-01409-21.svs
  📥 Copying slide to local SSD...
Slide: C3N-01409-21.svs

  Base magnification: 20.0x (openslide objective-power)
  Target magnification: 5.0x
  Required downsample: 4.0
  Chosen level: 1
  Level downsample: 4.000167086463476
  Direct magnification at chosen level: 4.9998x
  Target field-of-view at level 0: 2048 x 2048
  Read size at chosen level before resize: 512 x 512
  Final output patch size: 512 x 512
  Level dimensions: ((51791, 39041), (12947, 9760), (3236, 2440))
  Level downsamples: (1.0, 4.000167086463476, 16.002522594176174)

  Slide dimensions (level 0): 51791 x 39041
  Thumbnail downsample used for tissue mask: 32
  Thumbnail step: 64
  Approx max non-overlapping grid at target FOV: 25 x 19

    🔍 Found 214 valid